# EDA: датасет `lenta` из scikit-uplift

Датасет лента: SMS/Viber-рассылка. Таргет — `response_att` (отклик), тритмент — `group` (`test` / `control`).
Задача: проверить пригодность данных для uplift-модели — качество, корректность рандомизации, средний эффект и его гетерогенность.

Содержание: 1) размер и качество данных · 2) тритмент, таргет, ATE (доля откликов в тесте vs в контроле) с доверительным интервалом · 3) проверка рандомизации (Standardized Mean Difference - нормированная разница средних значений признака)· 4) признаки: связь с таргетом и корреляции · 5) гетерогенность эффекта по сегментам · 6) выводы

In [1]:
!pip install scikit-uplift pandas numpy scipy plotly nbformat optuna lightgbm

In [2]:
import os
import warnings

warnings.filterwarnings("ignore", message=".*IProgress not found.*")

import numpy as np
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats as st
from pathlib import Path
from sklift.datasets import fetch_lenta, get_data_dir

pio.renderers.default = "notebook_connected"
pd.set_option("display.width", 200, "display.max_columns", 40)

TARGET, TREAT = "response_att", "group"
LEAK = ["response_sms", "response_viber"]   # см. раздел 3: это альтернативные таргеты, а не признаки
BLUE, RED, GREY, GREEN = "#2f5fa8", "#c23b3b", "#8a8f99", "#2e7d5b"
LAY = dict(template="plotly_white", font=dict(size=12), showlegend=False,
           margin=dict(l=60, r=20, t=50, b=50))
DATA_HOME = os.environ.get("SKLIFT_DATA_HOME") or None   # None = кэш по умолчанию

In [3]:
import pandas as pd
from sklift.datasets import fetch_lenta

data, y, t_raw = fetch_lenta(data_home=DATA_HOME, return_X_y_t=True)

dtypes = {c: ("float32" if pd.api.types.is_numeric_dtype(data[c]) else "category") for c in data.columns}
X = data.astype(dtypes)
y = y.astype("int8")
num = X.select_dtypes("number").columns.tolist()

# True = тритмент; работает и для строк ("test"/"control"), и для 0/1
t = (t_raw == "test").to_numpy() if not pd.api.types.is_numeric_dtype(t_raw) else (t_raw == 1).to_numpy()
yv = y.to_numpy(float)
n, nt, nc = len(X), int(t.sum()), int((~t).sum())

print(f"строк: {n:,} | признаков: {X.shape[1]} (числовых {len(num)}, категориальных {X.shape[1]-len(num)})")
print(f"таргет: {y.name} | тритмент: {t_raw.name} -> "
      f"{ {k: int(v) for k, v in t_raw.value_counts().items()} }")
print(f"памяти под данные: {(X.memory_usage(deep=True).sum()+y.nbytes)/1e9:.2f} ГБ")
print("утечки в признаках:", [c for c in LEAK if c in X.columns])
X[["age", "gender", "main_format", "months_from_register", "cheque_count_12m_g41"]].head(3)

строк: 687,029 | признаков: 193 (числовых 192, категориальных 1)
таргет: response_att | тритмент: group -> {'test': 515892, 'control': 171137}
памяти под данные: 0.53 ГБ
утечки в признаках: ['response_sms', 'response_viber']


,age,gender,main_format,months_from_register,cheque_count_12m_g41
0,47.0,Ж,0.0,18.0,6.0
1,57.0,Ж,1.0,4.0,1.0
2,38.0,М,0.0,34.0,14.0


## 1. Размер и качество данных

In [4]:
miss = X[num].isna().mean()
nan_per_row = X[num].isna().sum(axis=1).to_numpy()
desc = X[num].agg(["mean", "std", "min", "median", "max"]).T
desc["p99"] = X[num].quantile(0.99)
desc["miss"] = miss
desc["nunique"] = X[num].nunique()
desc["tail_p99_to_max"] = desc["max"] / desc["p99"].replace(0, np.nan)

quality = pd.Series({
    "строк": n,
    "признаков всего": X.shape[1],
    "дублей по вектору признаков": int(X.duplicated().sum()),
    "признаков с пропусками": int((miss > 0).sum()),
    "признаков с >90% NaN": int((miss > 0.9).sum()),
    "признаков без NaN": int((miss == 0).sum()),
    "строк без пропусков": round(float((nan_per_row == 0).mean()), 4),
    "константных признаков": int((desc["nunique"] <= 1).sum()),
    "признаков с отрицательными значениями": int((desc["min"] < 0).sum()),
    
}, dtype=object)
display(quality.to_frame("значение"))
print("\nсамый тяжёлый правый хвост (max / p99):")
display(desc.sort_values("tail_p99_to_max", ascending=False)[["median", "p99", "max", "miss"]].head(5).round(2))
print("\nвозраст: доля < 16 лет = %.4f, доля > 90 = %.5f, доля NaN = %.4f"
      % ((X.age < 16).mean(), (X.age > 90).mean(), X.age.isna().mean()))
# блоки признаков с одинаковой долей пропусков -> общий источник выгрузки
blocks = miss.round(5).value_counts()
print("\nблоки с одинаковой долей NaN (доля NaN -> сколько признаков):")
print(blocks[blocks >= 3].head(5).to_dict())

,значение
строк,687029
признаков всего,193
дублей по вектору признаков,30
признаков с пропусками,152
признаков с >90% NaN,0
признаков без NaN,40
строк без пропусков,0.0082
константных признаков,0
признаков с отрицательными значениями,0



самый тяжёлый правый хвост (max / p99):


,median,p99,max,miss
sale_sum_12m_g27,425.820007,11810.57,2.866854e+06,0.00
sale_count_6m_g44,3.000000,82.00,1.172100e+04,0.00
cheque_count_3m_g79,0.000000,5.00,4.830000e+02,0.14
cheque_count_6m_g79,0.000000,9.00,8.070000e+02,0.00
sale_count_6m_g24,5.240000,172.03,1.526136e+04,0.00



возраст: доля < 16 лет = 0.0027, доля > 90 = 0.00019, доля NaN = 0.0171

блоки с одинаковой долей NaN (доля NaN -> сколько признаков):
{0.0: 41, 0.00453: 32, 0.13691: 16, 0.01249: 4, 0.19896: 4}


In [5]:
f = make_subplots(rows=3, cols=1, vertical_spacing=0.1,
                  subplot_titles=("Доля пропусков по признакам", "Топ-10 признаков по пропускам",
                                  "Пропусков в профиле одного клиента"))
f.add_histogram(x=miss * 100, nbinsx=30, marker_color=BLUE, row=1, col=1)
top = miss.sort_values(ascending=False).head(10)
f.add_bar(x=top.values * 100, y=list(top.index), orientation="h", marker_color=BLUE, row=2, col=1)
cnt, edges = np.histogram(nan_per_row, bins=30)
f.add_bar(x=(edges[:-1] + edges[1:]) / 2, y=cnt, marker_color=GREY, row=3, col=1)
f.update_yaxes(autorange="reversed", tickfont_size=9, row=2, col=1)
f.update_xaxes(title_text="% пропусков", row=1, col=1); f.update_xaxes(title_text="%", row=2, col=1)
f.update_xaxes(title_text="шт. из %d" % len(num), row=3, col=1)
f.update_yaxes(title_text="признаков", row=1, col=1); f.update_yaxes(title_text="клиентов", row=3, col=1)
f.update_layout(height=1000, **LAY)
f.update_layout(margin_l=260)   # место под полные имена признаков
f.show()

**Выводы по качеству данных.**
- Только 40 признаков не содержат пропуски.
- Много признаков содержат пропуски. Но число пропусков до 70 % - вероятнее всего, это информативные пропуски.
- Часть признаков группируются в блоки с идентичной долей пропусков. Вероятно, это признаки с одинаковой причиной пропуска в данных.
- Денежные и счётные признаки сильно скошены (например, у sale_count_6m_g44 медиана = 3, р99 = 82). Нужно будет обрабатывать эти выбросы.

## 2. Тритмент, таргет и средний эффект (ATE)

In [6]:
p_t, p_c = yv[t].mean(), yv[~t].mean()
ate = p_t - p_c
se = np.sqrt(p_t * (1 - p_t) / nt + p_c * (1 - p_c) / nc)
lo, hi = ate - 1.96 * se, ate + 1.96 * se

print(f"test: {p_t:.4f} (n={nt:,}) | control: {p_c:.4f} (n={nc:,})")
print(f"ATE = {ate}   95% ДИ [{lo}; {hi}]   "
      f"{'значим' if lo > 0 or hi < 0 else 'не значим'} (0 {'вне' if lo > 0 or hi < 0 else 'внутри'} ДИ)")

test: 0.1101 (n=515,892) | control: 0.1026 (n=171,137)
ATE = 0.0075471269267721325   95% ДИ [0.005874949095901839; 0.009219304757642426]   значим (0 вне ДИ)


In [7]:
f = make_subplots(rows=1, cols=3, column_widths=[0.28, 0.36, 0.36],
                  subplot_titles=("Размер групп, тыс.", "Доля отклика, % (95% ДИ)", "ATE, п.п."))
f.add_bar(x=["test", "control"], y=[nt / 1e3, nc / 1e3], marker_color=[BLUE, GREY],
          text=[f"{nt:,}<br>{nt/n:.1%}", f"{nc:,}<br>{nc/n:.1%}"], textposition="inside", row=1, col=1)
f.add_bar(x=["test", "control"], y=[p_t * 100, p_c * 100], marker_color=[BLUE, GREY],
          error_y=dict(type="data", color="#333",
                       array=[196 * np.sqrt(p_t * (1 - p_t) / nt), 196 * np.sqrt(p_c * (1 - p_c) / nc)]),
          text=[f"{p_t:.2%}", f"{p_c:.2%}"], textposition="outside", row=1, col=2)
f.add_scatter(x=[ate * 100], y=["ATE"], mode="markers+text", marker=dict(size=13, color=RED),
              error_x=dict(type="data", array=[1.96 * se * 100], color=RED, thickness=2),
              text=[f"  {ate}"], textposition="top center", row=1, col=3)
f.add_vline(x=0, line_dash="dot", line_color=GREY, row=1, col=3)
f.update_yaxes(range=[0, 13], row=1, col=2)
f.update_xaxes(range=[-0.1, 1.2], row=1, col=3); f.update_yaxes(showticklabels=False, row=1, col=3)
f.update_layout(height=330, **LAY)
f.show()

**Выводы по эффекту.** Рассылка работает, но эффект от нее мал: +0.00755 при базовом отклике 0.1026%;
доверительный интервал [+0.587; +0.922] п.п. далёк от нуля. Группы несбалансированы по размеру: 75% / 25%.

## 3. Проверка рандомизации

Нужно проверить, что признаки слабо отделяют treatment и control.

In [8]:
def smd_table(frame, treat_mask, cols):
    A = frame[cols].to_numpy(np.float32)
    out = []
    for j, c in enumerate(cols):
        x = A[:, j].astype(float); ok = ~np.isnan(x)
        a, b = x[ok & treat_mask], x[ok & ~treat_mask]
        sd = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
        out.append(dict(feature=c, smd=(a.mean() - b.mean()) / sd if sd > 0 else 0.0,
                        miss_diff=np.isnan(x[treat_mask]).mean() - np.isnan(x[~treat_mask]).mean()))
    return pd.DataFrame(out)

B = smd_table(X, t, num)
clean = B[~B.feature.isin(LEAK)]
print(f"|SMD| > 0.1 без утечек: {int((clean.smd.abs() > 0.1).sum())} из {len(clean)}; "
      f"max |SMD| = {clean.smd.abs().max():.3f}; "
      f"макс. расхождение пропусков = {clean.miss_diff.abs().max()*100:.2f} п.п.")
display(clean.reindex(clean.smd.abs().sort_values(ascending=False).index).head(6).round(4).set_index("feature"))

|SMD| > 0.1 без утечек: 0 из 190; max |SMD| = 0.025; макс. расхождение пропусков = 0.73 п.п.


,smd,miss_diff
feature,,
k_var_count_per_cheque_1m_g34,0.0253,-0.0038
crazy_purchases_cheque_count_12m,0.0249,0.0000
crazy_purchases_goods_count_12m,0.0244,0.0000
crazy_purchases_cheque_count_6m,0.0232,0.0000
k_var_disc_share_1m_g34,0.0224,-0.0037
crazy_purchases_cheque_count_3m,0.0216,0.0000


In [9]:
f = make_subplots(rows=3, cols=1, vertical_spacing=0.1,
                  subplot_titles=("|SMD| по всем признакам", "Топ-10 по |SMD|",
                                  "Разница доли пропусков test − control"))
f.add_histogram(x=B.smd.abs(), nbinsx=40, marker_color=BLUE, row=1, col=1)
f.add_vline(x=0.1, line_dash="dash", line_color=RED, annotation_text="порог 0.1", row=1, col=1)
tp = B.reindex(B.smd.abs().sort_values(ascending=False).index).head(10)
f.add_bar(x=tp.smd.abs(), y=list(tp.feature), orientation="h",
          marker_color=[RED if s in LEAK else BLUE for s in tp.feature], row=2, col=1)
f.add_vline(x=0.1, line_dash="dash", line_color=RED, row=2, col=1)
f.add_histogram(x=B.miss_diff * 100, nbinsx=40, marker_color=GREY, row=3, col=1)
f.update_yaxes(autorange="reversed", tickfont_size=9, row=2, col=1)
f.update_yaxes(title_text="признаков", row=1, col=1); f.update_xaxes(title_text="|SMD|", row=1, col=1)
f.update_yaxes(title_text="признаков", row=3, col=1); f.update_xaxes(title_text="п.п.", row=3, col=1)
f.update_layout(height=1000, **LAY)
f.update_layout(margin_l=260)
f.show()

### Проверка рандомизации классификатором

SMD смотрит на каждый признак по отдельности и не увидит дисбаланс, зашитый в комбинацию признаков. Прямой тест: обучить
классификатор `test` против `control`. При корректной рандомизации AUC должен быть равен 0.5. Признаки, которые сами по себе
хорошо делят группы, из модели исключаем.

In [10]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

A = X[num].to_numpy(np.float32)      # матрица числовых признаков, переиспользуется ниже


def auc_rank(x, pos):
    # AUC признака x как «предсказания» бинарной метки pos; NaN исключаются
    ok = ~np.isnan(x)
    xx, pp = x[ok], pos[ok]
    n1, n0 = int(pp.sum()), int((~pp).sum())
    if min(n1, n0) < 10 or np.all(xx == xx[0]):
        return np.nan
    r = st.rankdata(xx)
    return (r[pp].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


rs = np.random.default_rng(0)
sub_auc = rs.choice(n, 150_000, replace=False)                    # для univariate AUC подвыборки достаточно
auc_treat = pd.Series([auc_rank(A[sub_auc, j].astype(float), t[sub_auc]) for j in range(len(num))], index=num)
sep = (auc_treat - 0.5).abs().sort_values(ascending=False)
print("признаки, различающие test/control сами по себе (|AUC - 0.5| > 0.01):", sep[sep > 0.01].index.tolist())
display(pd.DataFrame({"auc_treat": auc_treat[sep.index[:6]],
                      "smd": B.set_index("feature").smd.reindex(sep.index[:6])}).round(4))

признаки, различающие test/control сами по себе (|AUC - 0.5| > 0.01): ['response_sms', 'response_viber']


,auc_treat,smd
response_sms,0.5413,0.1979
response_viber,0.5165,0.0685
k_var_disc_share_15d_g34,0.5090,0.0157
k_var_disc_share_3m_g34,0.5090,0.0171
crazy_purchases_cheque_count_12m,0.5090,0.0249
crazy_purchases_goods_count_12m,0.5089,0.0244


In [11]:
import optuna, lightgbm as lgb
optuna.logging.set_verbosity(optuna.logging.WARNING)

SUB = 150_000
sub = rs.choice(n, SUB, replace=False)
feat_clean = [c for c in num if c not in LEAK]
cv = StratifiedKFold(3, shuffle=True, random_state=42)

def tune(cols, target, n_trials=100):
    Xs, ys = X.iloc[sub][cols], target[sub]
    def obj(tr):
        m = lgb.LGBMClassifier(
            n_estimators=tr.suggest_int("n_estimators", 100, 600),
            learning_rate=tr.suggest_float("learning_rate", 0.01, 0.2, log=True),
            num_leaves=tr.suggest_int("num_leaves", 8, 128, log=True),
            min_child_samples=tr.suggest_int("min_child_samples", 20, 500, log=True),
            subsample=tr.suggest_float("subsample", 0.5, 1), subsample_freq=1,
            colsample_bytree=tr.suggest_float("colsample_bytree", 0.5, 1),
            reg_lambda=tr.suggest_float("reg_lambda", 1e-3, 10, log=True),
            n_jobs=1, verbose=-1, random_state=42)
        return cross_val_score(m, Xs, ys, cv=cv, scoring="roc_auc").mean()
    s = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    s.optimize(obj, n_trials=n_trials, n_jobs=8)
    return s.best_value

print(f"AUC без post-treatment (CV, лучший триал) = {tune(feat_clean, t):.4f}")

AUC без post-treatment (CV, лучший триал) = 0.5094


Рандомизация корректная, качество модели, разделяющей тритмент и контроль минимальное. SMD у всех признаков без утечки не сильно выше 0.1. Единственное что настораживает слишком разное число пропусков по некоторым признакам в тритмент и контроль.

## 4. Признаки: связь с таргетом и взаимные корреляции

Корреляции считаем **внутри контроля**, чтобы эффект кампании не влиял на оценку прогностической силы.

In [13]:
Xc = X.loc[~t, num]
yc = pd.Series(yv[~t], index=Xc.index)
corr_ctrl = Xc.corrwith(yc).rename("corr_y_ctrl")
corr_tbl = pd.concat([corr_ctrl, miss.rename("miss"), B.set_index("feature").smd], axis=1)
top_prog = corr_ctrl.abs().sort_values(ascending=False).head(12).index
display(corr_tbl.loc[top_prog].round(4))

,corr_y_ctrl,miss,smd
k_var_days_between_visits_15d,0.3007,0.1120,0.0117
k_var_disc_per_cheque_15d,0.2943,0.1988,0.0135
k_var_days_between_visits_1m,0.2811,0.1337,0.0136
k_var_cheque_15d,0.2779,0.1990,0.0139
stdev_discount_depth_15d,0.2631,0.2044,0.0158
k_var_sku_per_cheque_15d,0.2577,0.1990,0.0138
k_var_cheque_category_width_15d,0.2561,0.1990,0.0128
k_var_cheque_group_width_15d,0.2533,0.1990,0.0139
k_var_discount_depth_15d,0.2528,0.2044,0.0135
cheque_count_12m_g41,0.2318,0.0000,0.0116


In [15]:
def bins_of(s, nb=5):
    # квантильные бины по непропущенным значениям; NaN — отдельная упорядоченная категория
    if s.notna().sum() == 0:
        return pd.Categorical(["NaN"] * len(s), categories=["NaN"], ordered=True)
    q = np.unique(s.quantile(np.linspace(0, 1, nb + 1)).dropna().to_numpy())
    if q.size < 3:                                    # мало уникальных значений — бин на значение
        vals = np.sort(s.dropna().unique())[:nb]
        lab = s.map(lambda v: "NaN" if pd.isna(v) else f"{v:g}")
        cats = [f"{v:g}" for v in vals] + ["NaN"]
    else:
        cut = pd.cut(s, bins=q, include_lowest=True)
        lab = cut.astype(str).where(s.notna(), "NaN")
        cats = [str(c) for c in cut.cat.categories] + ["NaN"]
    return pd.Categorical(lab, categories=cats, ordered=True)


show = ["k_var_days_between_visits_3m", "cheque_count_12m_g41", "food_share_1m", "age", "months_from_register"]
f = make_subplots(rows=1, cols=2, column_widths=[0.42, 0.58],
                  subplot_titles=("Корреляция с таргетом (контроль)", "Доля отклика в контроле по бинам, %"))
tp = corr_ctrl.loc[top_prog[:10]]
f.add_bar(x=tp.values, y=[s[:30] for s in tp.index], orientation="h", marker_color=BLUE, row=1, col=1)
f.update_yaxes(autorange="reversed", tickfont_size=9, row=1, col=1)
for c, col in zip(show, [BLUE, RED, GREEN, "#b58900", "#6a4c93"]):
    rr = yc.groupby(bins_of(Xc[c]), observed=True).mean()
    rr = rr[rr.index != "NaN"]
    f.add_scatter(x=np.arange(1, len(rr) + 1), y=rr.to_numpy() * 100, mode="lines+markers", name=c[:28],
                  line_color=col, showlegend=True, row=1, col=2)
f.add_hline(y=p_c * 100, line_dash="dot", line_color=GREY, row=1, col=2)
f.update_xaxes(title_text="бин признака по возрастанию", dtick=1, row=1, col=2)
f.update_layout(height=380, **{**LAY, "showlegend": True, "margin": dict(l=60, r=20, t=50, b=95),
                               "legend": dict(x=0.46, y=-0.3, orientation="h", font_size=9)})
f.show()

In [16]:
corr_m = X[top_prog.tolist() + ["age", "months_from_register"]].corr()
short = [s.replace("k_var_", "kv_").replace("cheque_", "chq_")[:24] for s in corr_m.columns]
pairs = corr_m.to_numpy()[np.triu_indices(len(corr_m), 1)]
print(f"средний |r| между топ-признаками = {np.abs(pairs).mean():.2f}; пар с |r| > 0.9: {int((np.abs(pairs) > 0.9).sum())}")
f = go.Figure(go.Heatmap(z=corr_m.to_numpy(), x=short, y=short, zmin=-1, zmax=1,
                         colorscale="RdBu", reversescale=True, colorbar=dict(thickness=12)))
f.update_xaxes(tickangle=50, tickfont_size=8); f.update_yaxes(tickfont_size=8)
f.update_layout(title="Корреляции наиболее прогностичных признаков", height=560, width=760,
                **{**LAY, "margin": dict(l=130, r=10, t=50, b=130)})
f.show()

средний |r| между топ-признаками = 0.43; пар с |r| > 0.9: 6


### Систематический скрининг всех признаков: дубликаты, near-constant, univariate-сила

Корреляционная матрица выше построена только для топ-признаков. Здесь — сплошной скрининг: доля моды (near-constant),
univariate AUC и Gini по таргету для каждого признака, парный поиск дубликатов по |r| > 0.95 и жадный выбор представителя
кластера (оставляем признак с наибольшим |Gini|).

In [17]:
# 1. near-constant: доля самого частого значения (NaN считается значением)
d_idx = rs.choice(n, 100_000, replace=False)                      # подвыборка для скрининга
mode_share = X[num].iloc[d_idx].apply(lambda s: s.value_counts(normalize=True, dropna=False).iloc[0])
near_const = mode_share[mode_share > 0.99].index.tolist()

# 2. univariate-сила каждого признака по таргету, внутри контроля (без влияния кампании)
Ac, yc_bool = A[~t], yv[~t] > 0
auc_y = pd.Series([auc_rank(Ac[:, j].astype(float), yc_bool) for j in range(len(num))], index=num)
gini = (2 * auc_y - 1).rename("gini")
weak = gini[gini.abs() < 0.02].index.tolist()

# 3. дубликаты: корреляции по всем парам на подвыборке с медианной импутацией
Md = A[d_idx].copy()
Md = np.where(np.isnan(Md), np.nanmedian(Md, axis=0), Md)
sd = Md.std(0)
ok_cols = [c for c, k in zip(num, sd > 0) if k]
Z = (Md[:, sd > 0] - Md[:, sd > 0].mean(0)) / sd[sd > 0]
R = (Z.T @ Z) / len(Z)
iu = np.triu_indices(len(ok_cols), 1)
pairs = pd.DataFrame({"a": np.array(ok_cols)[iu[0]], "b": np.array(ok_cols)[iu[1]], "r": R[iu]})
dups = pairs[pairs.r.abs() > 0.95].reindex(pairs.r.abs().sort_values(ascending=False).index).dropna()

adj = {}
for a, b, _ in dups.itertuples(index=False):
    adj.setdefault(a, set()).add(b)
    adj.setdefault(b, set()).add(a)
drop_dup = set()
for c in gini.abs().sort_values(ascending=False).index:      # от информативных к шумным
    if c in drop_dup:
        continue
    drop_dup |= {q for q in adj.get(c, ()) if q not in drop_dup}

print(f"near-constant (доля моды > 99%): {len(near_const)} -> {near_const}")
print(f"пар с |r| > 0.95: {len(dups)}; признаков-дубликатов к удалению: {len(drop_dup)}")
print(f"слабых по univariate Gini (|Gini| < 0.02): {len(weak)}")
print(f"\nтоп-5 по univariate силе (контроль):")
display(pd.concat([auc_y.rename("auc_y"), gini, mode_share.rename("mode_share")], axis=1)
        .reindex(gini.abs().sort_values(ascending=False).index).head(5).round(4))
print("примеры найденных дубликатов:")
display(dups.head(6).round(3).set_index("a"))

near-constant (доля моды > 99%): 0 -> []
пар с |r| > 0.95: 4; признаков-дубликатов к удалению: 3
слабых по univariate Gini (|Gini| < 0.02): 4

топ-5 по univariate силе (контроль):


,auc_y,gini,mode_share
k_var_disc_per_cheque_15d,0.7448,0.4895,0.5188
k_var_cheque_15d,0.7419,0.4839,0.5187
k_var_days_between_visits_1m,0.7368,0.4735,0.6133
k_var_discount_depth_15d,0.7338,0.4676,0.5440
stdev_discount_depth_15d,0.7334,0.4669,0.5440


примеры найденных дубликатов:


,b,r
a,,
mean_discount_depth_15d,promo_share_15d,0.981
k_var_cheque_category_width_15d,k_var_cheque_group_width_15d,0.975
k_var_cheque_category_width_15d,k_var_sku_per_cheque_15d,0.971
sale_sum_12m_g24,sale_sum_6m_g24,0.953


In [18]:
f = make_subplots(rows=3, cols=1, vertical_spacing=0.1,
                  subplot_titles=("Univariate Gini по таргету (контроль)", "Топ-10 по |Gini|",
                                  "Максимальная |r| с другим признаком"))
f.add_histogram(x=gini.dropna(), nbinsx=40, marker_color=BLUE, row=1, col=1)
tp = gini.reindex(gini.abs().sort_values(ascending=False).index).head(10)
f.add_bar(x=tp.values, y=list(tp.index), orientation="h", marker_color=BLUE, row=2, col=1)
rmax = np.abs(R - np.eye(len(R))).max(axis=1)
f.add_histogram(x=rmax, nbinsx=40, marker_color=GREY, row=3, col=1)
f.add_vline(x=0.95, line_dash="dash", line_color=RED, row=3, col=1)
f.update_yaxes(autorange="reversed", tickfont_size=9, row=2, col=1)
f.update_yaxes(title_text="признаков", row=1, col=1)
f.update_xaxes(title_text="Gini", row=1, col=1); f.update_xaxes(title_text="max |r|", row=3, col=1)
f.update_layout(height=1000, **LAY)
f.update_layout(margin_l=260)
f.show()

Из 191 признака 2 (response_sms, response_viber) относятся к post-treatment и в модель не идут. Остальные описывают клиента до кампании.

В контрольной группе сильнее всего с откликом связаны признаки частоты и разнообразия покупок за 15 дней: AUC до 0,745 (Gini 0,49). Связь с откликом у остальных признаков умеренная или слабая.

Сильно коррелируют друг с другом (|r| > 0,95) только 4 пары, например mean_discount_depth_15d и promo_share_15d. Из каждой пары оставлен один признак. Почти константных признаков нет 

## 5. Гетерогенность эффекта — в каких бакетах признаков эффект выше/ниже

Для каждого признака: квантильные бины (плюс отдельный бин `NaN`), в каждом бине uplift = доля отклика в тесте минус доля в контроле,
однородность uplift между бинами — Q-тест Кохрана, поправка на множественность BH.

Q-test:
- Считается средневзвешенный uplift с весами 1/SE^2
- Складываются взвешенные квадраты отклонений аплифта в бинах от средневзвешенного - получается Q статистика и она сравнивается с chi squared. 
- Поправка BH - поправка на множественные тестирования.

In [ ]:
def segment_uplift(labels, min_n=300):
    cat = pd.Categorical(labels)
    codes, k = cat.codes.astype(np.int64), len(cat.categories)
    ok = codes >= 0
    idx = codes[ok] * 4 + t[ok].astype(np.int64) * 2 + (yv[ok] > 0)
    cnt = np.bincount(idx, minlength=k * 4).reshape(k, 2, 2)
    n_c, n_t = cnt[:, 0].sum(1), cnt[:, 1].sum(1)
    with np.errstate(invalid="ignore", divide="ignore"):
        rr_c, rr_t = cnt[:, 0, 1] / n_c, cnt[:, 1, 1] / n_t
        se = np.sqrt(rr_t * (1 - rr_t) / n_t + rr_c * (1 - rr_c) / n_c)
    out = pd.DataFrame(dict(seg=[str(c) for c in cat.categories], n=n_c + n_t, n_t=n_t, n_c=n_c,
                            rr_t=rr_t, rr_c=rr_c, uplift=rr_t - rr_c, se=se))
    return out[(out.n_t >= min_n) & (out.n_c >= min_n)].reset_index(drop=True)


def cochran_q(seg):
    # Q-тест Кохрана на однородность uplift между бинами -> (Q, df, p)
    u, w = seg.uplift.to_numpy(), 1 / seg.se.to_numpy() ** 2
    Q = float((w * (u - (u * w).sum() / w.sum()) ** 2).sum())
    return Q, len(seg) - 1, float(st.chi2.sf(Q, len(seg) - 1))


rows, segs_cache = [], {}
for c in num:
    seg = segment_uplift(bins_of(X[c]))
    if len(seg) < 2:
        continue
    Q, dfree, p = cochran_q(seg)
    segs_cache[c] = seg
    rows.append(dict(feature=c, bins=len(seg), Q=Q, p=p,
                     u_min=seg.uplift.min(), u_max=seg.uplift.max()))
H = pd.DataFrame(rows).sort_values("Q", ascending=False)
order = np.argsort(H.p.to_numpy())
adj = np.minimum.accumulate((H.p.to_numpy()[order] * len(H) / np.arange(1, len(H) + 1))[::-1])[::-1]
H["p_bh"] = pd.Series(adj, index=H.index[order]).clip(0, 1)

print(f"проверено признаков: {len(H)}; значимо неоднородны (BH q < 0.05): {int((H.p_bh < 0.05).sum())}")
display(H.head(8).assign(u_min=lambda d: (d.u_min * 100).round(2), u_max=lambda d: (d.u_max * 100).round(2))
         .round({"Q": 1, "p": 4, "p_bh": 3}).set_index("feature"))
print("\nсамый большой разброс uplift — но без значимости (шум мелких бинов):")
display(H.reindex((H.u_max - H.u_min).sort_values(ascending=False).index).head(3)
         .assign(u_min=lambda d: (d.u_min * 100).round(2), u_max=lambda d: (d.u_max * 100).round(2))
         .round({"Q": 1, "p_bh": 2}).set_index("feature")[["bins", "Q", "p_bh", "u_min", "u_max"]])

/var/folders/c5/dwqcr0qd20l4qb3yl0dkxh180000gn/T/ipykernel_2112/2061956915.py:14: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  return pd.Categorical(lab, categories=cats, ordered=True)
/var/folders/c5/dwqcr0qd20l4qb3yl0dkxh180000gn/T/ipykernel_2112/2061956915.py:14: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  return pd.Categorical(lab, categories=cats, ordered=True)
/var/folders/c5/dwqcr0qd20l4qb3yl0dkxh180000gn/T/ipykernel_2112/2061956915.py:14: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  return pd.Categorical(lab, categories=cats, ordered=True)
/var/folders/c5/dwqcr0qd20l4qb3yl0dkxh180000gn

проверено признаков: 191; значимо неоднородны (BH q < 0.05): 17


,bins,Q,p,u_min,u_max,p_bh
feature,,,,,,
k_var_days_between_visits_3m,5,22.1,0.0002,0.09,1.23,0.019
k_var_disc_share_6m_g27,5,21.2,0.0003,0.33,1.50,0.019
response_sms,4,19.1,0.0003,-0.08,0.86,0.019
k_var_count_per_cheque_6m_g27,5,18.6,0.0009,0.31,1.12,0.040
k_var_sku_price_6m_g27,5,18.4,0.0010,0.37,1.22,0.040
k_var_sku_price_6m_g48,5,15.5,0.0037,0.33,1.29,0.048
food_share_1m,4,15.5,0.0015,0.29,1.07,0.046
sale_count_3m_g24,5,15.3,0.0040,0.30,1.13,0.048



самый большой разброс uplift — но без значимости (шум мелких бинов):


,bins,Q,p_bh,u_min,u_max
feature,,,,,
crazy_purchases_cheque_count_1m,5,2.5,0.77,0.03,2.11
cheque_count_6m_g41,5,15.2,0.05,-0.78,1.25
cheque_count_6m_g48,5,13.1,0.09,-0.78,1.25


In [ ]:
names = H.feature.head(4).tolist()
f = make_subplots(rows=2, cols=2, subplot_titles=[s[:34] for s in names], vertical_spacing=0.17)
for i, c in enumerate(names):
    seg = segs_cache[c]
    f.add_bar(x=seg.seg, y=seg.uplift * 100, error_y=dict(type="data", array=1.96 * seg.se * 100,
              color="#444", thickness=1), marker_color=[RED if u < 0 else BLUE for u in seg.uplift],
              row=i // 2 + 1, col=i % 2 + 1)
    f.add_hline(y=ate * 100, line_dash="dot", line_color=GREEN, row=i // 2 + 1, col=i % 2 + 1)
f.update_xaxes(tickfont_size=8); f.update_yaxes(title_text="uplift, п.п.", tickfont_size=9)
f.update_annotations(font_size=11)
f.update_layout(height=520, **LAY)
f.show()

In [ ]:
cuts = {
    "Пол": X.gender.astype(str),
    "Пропусков в профиле": bins_of(pd.Series(nan_per_row.astype(float), index=X.index), nb=4),
    "Возраст": bins_of(X.age),
    "Месяцев с регистрации": bins_of(X.months_from_register),
}
f = make_subplots(rows=2, cols=2, subplot_titles=list(cuts), vertical_spacing=0.17)
for i, (title, lab) in enumerate(cuts.items()):
    seg = segment_uplift(lab)
    Q, dfree, p = cochran_q(seg)
    print(f"{title}: Q = {Q:.1f} (df {dfree}), p = {p:.4f}; uplift от {seg.uplift.min()*100:+.2f} "
          f"до {seg.uplift.max()*100:+.2f} п.п.")
    f.add_bar(x=seg.seg, y=seg.uplift * 100, error_y=dict(type="data", array=1.96 * seg.se * 100,
              color="#444", thickness=1), marker_color=BLUE, row=i // 2 + 1, col=i % 2 + 1)
    f.add_hline(y=ate * 100, line_dash="dot", line_color=GREEN, row=i // 2 + 1, col=i % 2 + 1)
f.update_xaxes(tickfont_size=9); f.update_yaxes(title_text="uplift, п.п.", tickfont_size=9)
f.update_annotations(font_size=11)
f.update_layout(height=520, **LAY)
f.show()

Пол: Q = 0.2 (df 1), p = 0.6727; uplift от +0.74 до +0.81 п.п.
Пропусков в профиле: Q = 12.1 (df 3), p = 0.0071; uplift от +0.36 до +0.91 п.п.
Возраст: Q = 5.3 (df 5), p = 0.3829; uplift от +0.41 до +0.98 п.п.
Месяцев с регистрации: Q = 4.8 (df 5), p = 0.4386; uplift от +0.51 до +1.11 п.п.


**Выводы по гетерогенности.** Неоднородность есть, значимо неоднородны 17 признаков из 191,
и сосредоточена она в активности пользователей. Самый устойчивый и управляемый разрез — полнота профиля: у клиентов с малым
числом пропусков uplift порядка 0.9 п.п., у «пустых» профилей — около 0.4 п.п., то есть вдвое ниже.
Демографические разрезы пусты — пол, возраст и срок с регистрации эффект не модифицируют.

## 6. Выводы: признаки, сплит, метрики

**Рандомизация корректна.** 
Максимальный |SMD| равен 0.025 при пороге 0.1, поэтому тест и контроль сопоставимы. Эффект небольшой, но значимый. Отклик в тесте 11.01%, в контроле 10.26%, ATE = +0.76 п.п. (95% ДИ 0.59–0.92 п.п.).

**Гетерогенность слабая**. Uplift по сегментам колеблется примерно в пределах 0.4–0.9 п.п. вокруг ATE. После q-test значимы 17 из 191 признака.

**Исключить** 
- response_sms и response_viber: это альтернативные таргеты, то есть утечка.
- Убрать 2 признака с |r| > 0.95. Константных признаков нет.

Пропуски, скорее всего, информативны, их не нужно просто выбрасывать: у 152 признаков есть NaN, полных строк только 0.8%. Число пропусков в строке само различает uplift (p = 0.007).